### Mapa interactivo de oportunidades

In [42]:
!pip install folium branca


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [43]:
import pandas as pd
import folium
from folium.plugins import MarkerCluster
from pathlib import Path

BASE_DIR = Path("..")

df = pd.read_csv(BASE_DIR / "data/processed/gym_market_opportunity_albacete_growth.csv")

gyms = pd.read_csv(BASE_DIR / "data/raw/places_all_results_v4.csv")

df.head()

,municipio,poblacion_2025,gyms_google,fitness_x10k,lat,lon,radius_m,gyms_sample_names,gyms_google_new,fitness_x10k_new,catchment_pop_25km,real_market_potential,dist_albacete_km,opportunity_score,catchment_gyms_25km,opportunity_score_v2,growth_5y,growth_10y,market_type
0,Abengibre,753,37,491.367862,39.210505,-1.541444,4000,NaN,0.0,0.000000,27838,27838.000000,36.521899,27838.000000,23.0,1159.916667,-0.010512,-0.037084,Mercado emergente
1,Alatoz,504,28,555.555556,39.095050,-1.361995,4000,NaN,0.0,0.000000,16360,16360.000000,44.454498,16360.000000,13.0,1168.571429,-0.003953,-0.173770,Mercado emergente
2,Albacete,175400,53,3.021665,38.994398,-1.860173,18000,"967GYM; ALMA SERENA ( CENTRO DE YOGA, MEDITACI...",78.0,4.446978,188399,2384.797468,0.000000,715.439241,89.0,2093.322222,0.006103,0.019051,Mercado emergente
3,Albatana,644,46,714.285714,38.572000,-1.520560,4000,Circuito ALBATANA,1.0,15.527950,43803,21901.500000,55.430927,21901.500000,24.0,1752.120000,-0.051546,-0.134409,Oportunidad latente
4,Alborea,672,20,297.619048,39.279205,-1.395773,4000,Polideportivo Municipal de Alborea,1.0,14.880952,18660,9330.000000,51.060574,9330.000000,17.0,1036.666667,0.012048,-0.108753,Mercado competitivo


In [44]:
# creamos mapa + escala de colores
import folium
import branca.colormap as cm

center = [df["lat"].mean(), df["lon"].mean()]

m = folium.Map(location=center, zoom_start=9, tiles="cartodbpositron")

col = cm.linear.YlOrRd_09.scale(
    df["opportunity_score_v2"].min(),
    df["opportunity_score_v2"].max()
)
col.caption = "Opportunity score v2"
col.add_to(m)



In [45]:
# Capa 1: Municipios (círculos)
from folium import FeatureGroup

fg_munis = FeatureGroup(name="Municipios (oportunidad)", show=True)
m.add_child(fg_munis)

for _, r in df.iterrows():
    if pd.isna(r["lat"]) or pd.isna(r["lon"]):
        continue

    popup = f"""
    <b>{r['municipio']}</b><br>
    Población 2025: {int(r['poblacion_2025'])}<br>
    Gimnasios (Places): {int(r['gyms_google_new'])}<br>
    Fitness density: {round(r['fitness_x10k_new'],2)}<br>
    Growth 5y: {round(r['growth_5y']*100,2)}%<br>
    Growth 10y: {round(r['growth_10y']*100,2)}%<br>
    Opportunity score v2: <b>{round(r['opportunity_score_v2'],1)}</b>
    """

    radius = 4 + (r["poblacion_2025"] / 25000)

    folium.CircleMarker(
        location=[r["lat"], r["lon"]],
        radius=radius,
        color=col(r["opportunity_score_v2"]),
        fill=True,
        fill_color=col(r["opportunity_score_v2"]),
        fill_opacity=0.85,
        tooltip=f"{r['municipio']} | score {round(r['opportunity_score_v2'],1)}",
        popup=folium.Popup(popup, max_width=340)
    ).add_to(fg_munis)



In [46]:
# Capa 2: TOP 15 oportunidades (marcadores)
fg_top = FeatureGroup(name="TOP 15 oportunidades", show=True)
m.add_child(fg_top)

top15 = df.sort_values("opportunity_score_v2", ascending=False).head(15)

for _, r in top15.iterrows():
    folium.Marker(
        location=[r["lat"], r["lon"]],
        tooltip=f"TOP: {r['municipio']} ({round(r['opportunity_score_v2'],1)})",
        popup=f"<b>{r['municipio']}</b><br>Opportunity score v2: {round(r['opportunity_score_v2'],1)}"
    ).add_to(fg_top)



In [47]:
# capa 3: Gimnasios (MarkerCluster)
from folium.plugins import MarkerCluster

fg_gyms = FeatureGroup(name="Gimnasios (Google Places)", show=True)
m.add_child(fg_gyms)

cluster = MarkerCluster().add_to(fg_gyms)

for _, r in gyms.iterrows():
    if pd.isna(r["lat"]) or pd.isna(r["lon"]):
        continue

    tooltip = f"{r.get('name','Gym')} | {r.get('municipio','')}"
    popup = f"""
    <b>{r.get('name','Gym')}</b><br>
    Municipio: {r.get('municipio','')}<br>
    Types: {r.get('types','')}<br>
    Address: {r.get('formattedAddress','')}
    """

    folium.CircleMarker(
        location=[r["lat"], r["lon"]],
        radius=3,
        fill=True,
        fill_opacity=0.8,
        tooltip=tooltip,
        popup=folium.Popup(popup, max_width=350)
    ).add_to(cluster)



In [48]:
# controles + guardar html
from pathlib import Path

folium.LayerControl(collapsed=False).add_to(m)

out_path = Path("..") / "reports" / "outputs" / "mapa_final_oportunidades_fitness_albacete.html"
m.save(out_path)

print("Mapa guardado en:", out_path)



Mapa guardado en: ..\reports\outputs\mapa_final_oportunidades_fitness_albacete.html


In [49]:
# definir colores por tipo de mercado
market_colors = {
    "Mercado emergente": "green",
    "Mercado competitivo": "blue",
    "Oportunidad latente": "orange",
    "Mercado débil": "gray"
}

In [50]:
# crear capa de calificación de mercado
from folium import FeatureGroup

fg_market = FeatureGroup(name="Tipos de mercado", show=True)
m.add_child(fg_market)

for _, r in df.iterrows():
    
    if pd.isna(r["lat"]) or pd.isna(r["lon"]):
        continue
    
    color = market_colors.get(r["market_type"], "black")
    
    popup = f"""
    <b>{r['municipio']}</b><br>
    Tipo de mercado: <b>{r['market_type']}</b><br>
    Población: {int(r['poblacion_2025'])}<br>
    Growth 5y: {round(r['growth_5y']*100,2)}%<br>
    Opportunity score: {round(r['opportunity_score_v2'],1)}
    """
    
    folium.CircleMarker(
        location=[r["lat"], r["lon"]],
        radius=6,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.8,
        tooltip=r["municipio"],
        popup=popup
    ).add_to(fg_market)

m

In [51]:
# ================================
# MAPA DE TIPOS DE MERCADO
# ================================


In [52]:
# recargamos dataset y generamos mapa nuevo!
import folium

df = pd.read_csv(BASE_DIR / "data/processed/gym_market_opportunity_albacete_growth.csv")

center = [df["lat"].mean(), df["lon"].mean()]

m_market = folium.Map(
    location=center,
    zoom_start=9,
    tiles="cartodbpositron"
)

In [53]:
# definimos colores por tipo de mercado
market_colors = {
    "Mercado emergente": "green",
    "Mercado competitivo": "blue",
    "Oportunidad latente": "orange",
    "Mercado débil": "gray"
}

In [54]:
# agrego al mapa los municipios coloreados por tipo de mercado
from folium import FeatureGroup

fg_market = FeatureGroup(name="Tipos de mercado", show=True)
m_market.add_child(fg_market)

for _, r in df.iterrows():
    if pd.isna(r["lat"]) or pd.isna(r["lon"]):
        continue

    color = market_colors.get(r["market_type"], "black")

    popup = f"""
    <b>{r['municipio']}</b><br>
    Tipo de mercado: <b>{r['market_type']}</b><br>
    Población 2025: {int(r['poblacion_2025'])}<br>
    Growth 5y: {round(r['growth_5y'] * 100, 2)}%<br>
    Opportunity score v2: {round(r['opportunity_score_v2'], 1)}
    """

    folium.CircleMarker(
        location=[r["lat"], r["lon"]],
        radius=6,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.8,
        tooltip=f"{r['municipio']} | {r['market_type']}",
        popup=folium.Popup(popup, max_width=320)
    ).add_to(fg_market)

In [55]:
# agregamos la leyenda personalizada
legend_html = """
<div style="
position: fixed;
bottom: 40px;
left: 40px;
width: 220px;
height: 130px;
background-color: white;
border: 2px solid grey;
z-index: 9999;
font-size: 14px;
padding: 10px;
">
<b>Tipos de mercado</b><br><br>
<span style="color:green;">●</span> Mercado emergente<br>
<span style="color:blue;">●</span> Mercado competitivo<br>
<span style="color:orange;">●</span> Oportunidad latente<br>
<span style="color:gray;">●</span> Mercado débil
</div>
"""

m_market.get_root().html.add_child(folium.Element(legend_html))

In [56]:
# guardo el mapa
from pathlib import Path

folium.LayerControl(collapsed=False).add_to(m_market)

out_path = Path("..") / "reports" / "outputs" / "mapa_tipos_mercado_fitness_albacete.html"
m_market.save(out_path)

print("Mapa guardado en:", out_path)


Mapa guardado en: ..\reports\outputs\mapa_tipos_mercado_fitness_albacete.html


### Objetivo: crear mapas interactivos del mercado fitness.

Contenido:

Mapa 1 — Mapa de oportunidades

municipios coloreados por opportunity_score

top 15 oportunidades

gimnasios existentes

Resultado:

mapa_final_oportunidades_fitness_albacete.html

Mapa 2 — Mapa estratégico de mercados

clasificación por tipo de mercado

leyenda de colores

Resultado:

mapa_tipos_mercado_fitness_albacete.html